# E10b · Variables lingüísticas nuevas a partir del análisis de errores
**Parte de:** E0 (stacking de 5 modelos, CV 0,8126) y E8b (bagging de 5 semillas con los 5 modelos + LightGBM, CV 0,8179, público 0,80168).

**Hipótesis B.** En el notebook original, el bloque de léxico hecho a mano (H1-H10) fue lo que más aportó (+2,2 puntos), mientras que un léxico genérico (HurtLex, E5) empeoró. Si el valor está en patrones de **contexto** escritos a partir de leer el corpus, repetir el ejercicio sobre los **errores** del mejor modelo debería encontrar patrones nuevos que el modelo todavía no capta.

**Procedimiento.**
1. Se leen los falsos positivos y negativos OOF de E8b por etiqueta (solo train).
2. Se formulan hipótesis nuevas (H12-H17) y se verifican como en la sección 6.3 del notebook original: P(etiqueta | patrón) frente a la prevalencia, con n.
3. Las que se sostienen se agregan al bloque de léxico (`extra_lex`) y se mide la CV repetida.
4. Control del **sesgo de selección**: los patrones se escribieron mirando los errores del mismo train con que se evalúan, así que la CV es optimista. Se miden una versión "genérica" sin palabras copiadas de tweets concretos y la frecuencia de los patrones en el test (solo conteos, sin etiquetas).

El código de las variables está en `ext_variables.py`.

### Glosario de esta prueba
| Término | Significado |
|---|---|
| **HS / TR / AG** | Las 3 etiquetas: discurso de odio / odio dirigido a una persona concreta / agresividad |
| **F1 medio** | F1 de la clase 1 de cada etiqueta, promediado entre HS, TR y AG (métrica principal) |
| **CV repetida** | Validación cruzada de 5 particiones repetida con 3 semillas (42, 43, 44); se reporta media ± desviación |
| **OOF** | *Out-of-fold*: predicción de cada fila del train hecha por un modelo que no la vio al entrenar |
| **FP / FN** | Falso positivo (el modelo dice 1 y era 0) / falso negativo (dice 0 y era 1) |
| **Stacking** | Un metamodelo (regresión logística) aprende a combinar las probabilidades OOF de los modelos base |
| **Seed bagging** | Promediar modelos entrenados con distintas particiones aleatorias |
| **Ablación** | Quitar una variable a la vez para ver cuánto aporta |
| **Sesgo de selección** | Si las reglas se inventan mirando los errores de un conjunto, medirlas en ese mismo conjunto exagera su mérito |
| **Demostrativo** | *este, esta, esos...*: señala a alguien o algo concreto |
| **Posesivo** | *mi, su...*: indica pertenencia (*mi perra* = la mascota) |

El glosario completo está en `RESUMEN.md`.

In [1]:
import os, sys, platform, re
sys.path.insert(0, os.path.abspath('.'))          # el notebook vive en experimentos/, junto a toxic_base.py
import numpy as np, pandas as pd, sklearn, scipy
import toxic_base as tb
import ext_variables as ev
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30); pd.set_option('display.max_colwidth', 130)
print('Python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__,
      '| scikit-learn', sklearn.__version__, '| scipy', scipy.__version__)
print('Semillas de la CV repetida:', tb.SEEDS, '| particiones:', tb.N_FOLDS)
d = tb.Datos()
print('Datos en', d.data_dir, '| train', d.train.shape, '| test', d.test.shape)
S5, B5 = (42, 43, 44, 45, 46), tb.BASES_ORIGINALES
B6 = B5 + ['LGBM']

Python 3.12.3 | numpy 2.5.4 | pandas 3.0.6 | scikit-learn 1.9.1 | scipy 1.18.1
Semillas de la CV repetida: (42, 43, 44) | particiones: 5


Datos en /home/jc/repositories/natural-language-processing-workshops/workshops/Kaggle-tweet-toxic/tweet-toxic-2026-puj | train (1731, 5) | test (1154, 2)


## 1. Análisis de errores de E8b
Probabilidad OOF de E8b promediada sobre las 3 semillas del metamodelo, umbral 0,5. Se muestran los errores más seguros de cada tipo; en la exploración se leyeron todos (unos 290 de HS, 170 de TR y 260 de AG).

In [2]:
P8 = np.mean([tb.stacking_bagged(d, B6, S5, s, con_test=False)[0] for s in tb.SEEDS], 0)
Yp8 = tb.binarizar(P8)
print({k: round(v, 4) for k, v in tb.metricas(d.Y, Yp8).items()})
for j, lab in enumerate(tb.LABELS):
    fp = np.where((Yp8[:, j] == 1) & (d.Y[:, j] == 0))[0]
    fn = np.where((Yp8[:, j] == 0) & (d.Y[:, j] == 1))[0]
    print(f'\n===== {lab}: {len(fp)} falsos positivos, {len(fn)} falsos negativos')
    for tipo, idx in [('FP', fp[np.argsort(-P8[fp, j])][:8]), ('FN', fn[np.argsort(P8[fn, j])][:8])]:
        for i in idx:
            print(f'  {tipo} p={P8[i, j]:.2f} [{d.COMBO[i]}]  {d.TXT_TR.iloc[i][:125]}')

{'F1 HS': 0.8074, 'F1 TR': 0.8407, 'F1 AG': 0.8062, 'F1 medio': np.float64(0.8181), 'F1 micro': 0.816, 'F1-macro 2 clases': np.float64(0.8515), 'EMR': np.float64(0.7799)}

===== HS: 145 falsos positivos, 146 falsos negativos
  FP p=0.97 [000]  los asaltantes subsaharianos de la valla de ceuta estan bien entrenados militarmente. acaso en las fotos ve a muchas mujeres 
  FP p=0.96 [000]  racismo es que hagas enfrijoladas con frijoles geros y no negros.
  FP p=0.96 [000]  casado y rivera contra la inmigracin ilegal. snchez e iglesias a favor de la invasin masiva y violenta de subsaharianos.a qui
  FP p=0.96 [000]  y chilee?? cundo visitan a los sudacas xurl
  FP p=0.95 [000]  xuser xuser cllate zorra ac los leprosos no opinan
  FP p=0.95 [000]  xuser cllate todo te queda bien zorra te amo
  FP p=0.95 [000]  cllate negro ql ! cualquier numero 1!he sacado ms que voh ! t anda cuidarle la zorra a la gigi mejor mira que en cualquier mo
  FP p=0.94 [000]  xuser callate puta tu eres el traidor.


**Qué se observó al leer los errores.**
- **Falsos positivos de HS con *perra* como mascota o trato afectuoso**: *"no puedo creer lo que ronca mi perra"*, *"a mi perra me la tienen frita"*, *"le tengo que enseñar eso a mi perra"*. El posesivo *mi/su* delante del insulto casi nunca es odio (pero *"tú eres mi puta"* sí lo es).
- **Falsos positivos con el insulto negado o refutado para defender a las mujeres**: *"una tía no es una guarra por hacer una foto…"*, *"no es ninguna gata ni puta ni zorra"*, *"la mujer es una perla no una perra"*.
- **Falsos negativos con *árabe***: H7 dejó *árabe* como palabra mayormente benigna (canción, comida), y el modelo pasa por alto los estereotipos: *"el árabe es un flojo"*, *"estos árabes de mierda"*, *"malditos árabes"*, *"un árabe me está chanceando y siento que me va a tirar una bomba"*.
- **Falsos negativos de odio a inmigrantes sin las palabras de H5/H6**: deshumanización (*"fuera la escoria inmigrante"*, *"lacras sudacas"*, *"moromierdas delincuentes"*) o llamados a echar o violentar (*"sacar a patadas"*, *"que se extingan"*, *"un subfusil y me voy a la valla"*).
- **Demostrativo + grupo**: *"estos indocumentados"*, *"esos árabes"*, *"tus queridos refugiados"*: el demostrativo señala al grupo con desprecio.
- **TR sin mención**: el odio dirigido a una mujer concreta que no está mencionada se marca con un demostrativo (*"esta puta payasa"*, *"la perra esa"*, *"esta mina"*).
- Otros patrones que se miraron y se descartan abajo: titular de prensa sobre migración (FP), risas + insulto femenino, *hijos de puta* en plural (*"indios hijos de puta"*), estereotipo doméstico (*"a fregar"*, *"hazme la cena"*).

### Hipótesis nuevas
| # | Hipótesis | Variable (`ext_variables.py`) |
|---|---|---|
| H12 | El **posesivo** delante del insulto femenino (*mi perra, su perra*) señala a la mascota o un trato afectuoso: HS bajo. Se excluye *eres mi puta* | `posesivo_fem_sin_eres` |
| H13 | El insulto **negado o refutado** (*no es una guarra por…*, *ni puta ni zorra*) no es odio | `negacion_fem` |
| H14 | *Árabe* en un **estereotipo** (*estos/malditos árabes*, *árabes de mierda*, *bomba, terrorista, explotar*) sí es odio, al contrario de H7 | `arabe_hostil` |
| H15 | Tema migrante + **deshumanización** (*escoria, lacra, plaga, delincuentes*) o **expulsión/violencia** (*fuera, a patadas, que se mueran*) es odio agresivo y genérico (TR = 0) | `hostilidad_migrante`, `expulsion_migrante` |
| H16 | **Demostrativo o *putos* + grupo** en plural (*estos indocumentados*, *esos moros*) es odio genérico | `demostrativo_grupo` |
| H17 | Con HS = 1, un **demostrativo femenino** (*esta puta*, *la perra esa*, *esa vieja*) indica un blanco individual: TR alto aunque no haya mención | `demostrativo_fem` |

## 2. Verificación con datos (solo train)
Las columnas `pred` son la proporción que E8b predice como 1 en esas filas. Una variable es útil si la tasa real se aleja de la prevalencia **y** de lo que el modelo ya predice.

In [3]:
F = pd.DataFrame([ev.fila(t) for t in d.TXT_TR])
C = pd.DataFrame([ev.marcas(t) for t in d.TXT_TR])
T = d.TXT_TR
def tiene(p): return T.str.contains(p, regex=True).values
fem, arabe, migra = tiene(rf'\b{ev.FEM}\b'), tiene(r'\b(a)?rabes?\b'), C['_migra'].values
hs1 = d.Y[:, 0] == 1
mencion = d.train.text.str.match(r'^\W*@').values

def evidencia(mascara, hip, condicion, base=None):
    m = np.asarray(mascara, bool) if base is None else (np.asarray(mascara, bool) & base)
    return {'Hip.': hip, 'Condición': condicion, 'n': int(m.sum()),
            **{f'P({l})': d.Y[m, j].mean() for j, l in enumerate(tb.LABELS)},
            **{f'pred {l}': Yp8[m, j].mean() for j, l in enumerate(tb.LABELS)}}

filas = [
    evidencia(np.ones(d.n), '-', 'Todo el train (referencia)'),
    evidencia(fem, '-', 'Insulto femenino (referencia de H12-H13)'),
    evidencia(F.posesivo_fem_sin_eres, 'H12', 'mi/su + insulto femenino (sin "eres mi")'),
    evidencia(C.eres_mi_fem, 'H12', '"eres mi" + insulto femenino (excluido)'),
    evidencia(F.negacion_fem, 'H13', 'Insulto negado / refutado'),
    evidencia(arabe, '-', 'árabe (referencia de H14, = H7)'),
    evidencia(F.arabe_hostil, 'H14', 'árabe en estereotipo u hostilidad'),
    evidencia(migra, '-', 'Tema migrante (referencia de H15-H16)'),
    evidencia(F.hostilidad_migrante, 'H15', 'Migrante + deshumanización o expulsión/violencia'),
    evidencia(F.expulsion_migrante, 'H15', 'Migrante + expulsión/violencia'),
    evidencia(F.demostrativo_grupo, 'H16', 'estos/esos/putos + grupo en plural'),
    evidencia(F.demostrativo_fem, 'H17', 'Demostrativo femenino (esta puta, esa vieja)'),
    # candidatos descartados
    evidencia(migra & tiene(r'xurl\s*$') & ~tiene(r'\b(yo|me|mi|soy|tu|te|vos|eres)\b'), '✗', 'Titular: migrante + termina en URL, sin 1.ª/2.ª persona'),
    evidencia(fem & tiene(r'jaja|jeje|ajaj|jsjs|haha'), '✗', 'Risa + insulto femenino'),
    evidencia(tiene(r'\bhijos de (la |tu |su |bien )?(gran |mil |re )?(puta|perra)'), '✗', '"hijos de puta" en plural'),
    evidencia(tiene(r'\bfreg|\bcocin|\bla cena\b|\bplatos?\b|\bplanch|sandwich'), '✗', 'Estereotipo doméstico (fregar, cocina, la cena)'),
]
pd.DataFrame(filas).round(2)

,Hip.,Condición,n,P(HS),P(TR),P(AG),pred HS,pred TR,pred AG
0,-,Todo el train (referencia),1731,0.44,0.30,0.38,0.44,0.31,0.40
1,-,Insulto femenino (referencia de H12-H13),897,0.55,0.53,0.53,0.62,0.60,0.61
2,H12,"mi/su + insulto femenino (sin ""eres mi"")",23,0.13,0.13,0.17,0.48,0.48,0.52
3,H12,"""eres mi"" + insulto femenino (excluido)",12,0.75,0.75,0.67,0.83,0.83,0.83
4,H13,Insulto negado / refutado,13,0.46,0.31,0.31,0.85,0.62,0.77
5,-,"árabe (referencia de H14, = H7)",176,0.19,0.02,0.15,0.05,0.00,0.05
6,H14,árabe en estereotipo u hostilidad,44,0.57,0.02,0.39,0.16,0.00,0.11
7,-,Tema migrante (referencia de H15-H16),444,0.41,0.05,0.32,0.35,0.01,0.28
8,H15,Migrante + deshumanización o expulsión/violencia,29,0.93,0.03,0.86,0.55,0.00,0.48
9,H15,Migrante + expulsión/violencia,14,0.93,0.07,0.93,0.43,0.00,0.43


**H17 (y H15 para AG) condicionadas a HS = 1**, como H9 en el notebook original:

In [4]:
pd.DataFrame([evidencia(np.ones(d.n), '-', 'Todos los HS=1 (referencia)', hs1),
              evidencia(~mencion, '-', 'HS=1 sin @mención inicial', hs1),
              evidencia(F.demostrativo_fem, 'H17', 'Demostrativo femenino', hs1),
              evidencia(F.demostrativo_fem & ~mencion, 'H17', 'Demostrativo femenino sin @mención inicial', hs1),
              evidencia(F.hostilidad_migrante, 'H15', 'Migrante + deshumanización/expulsión', hs1)]
            ).drop(columns=['P(HS)', 'pred HS']).round(2)

,Hip.,Condición,n,P(TR),P(AG),pred TR,pred AG
0,-,Todos los HS=1 (referencia),756,0.69,0.84,0.60,0.75
1,-,HS=1 sin @mención inicial,395,0.46,0.76,0.34,0.62
2,H17,Demostrativo femenino,18,1.00,0.94,0.67,0.67
3,H17,Demostrativo femenino sin @mención inicial,10,1.00,1.00,0.50,0.50
4,H15,Migrante + deshumanización/expulsión,27,0.04,0.93,0.00,0.52


**Lectura de la evidencia.**
- **H12 se confirma**: con un insulto femenino P(HS) es 0,55, pero con *mi/su* delante cae a ≈ 0,13 y E8b predice HS en casi la mitad de esas filas. *Eres mi puta* va en sentido contrario (≈ 0,75), por eso se excluye.
- **H13 es débil** (13 tweets): P(HS) = 0,46, algo por debajo del 0,55 del insulto femenino, pero E8b predice HS en el 85 % de esas filas. La señal está en el error del modelo más que en la tasa.
- **H14 se confirma**: *árabe* en general tiene P(HS) ≈ 0,19, pero en un estereotipo sube a ≈ 0,5-0,6, y E8b casi nunca lo predice.
- **H15 y H16 se confirman con fuerza**: P(HS) ≈ 0,9 y P(AG) ≈ 0,85, con TR ≈ 0, frente a ≈ 0,4 en el tema migrante en general.
- **H17 se confirma**: con HS = 1, el demostrativo femenino da TR = 1 en todos los casos, frente a 0,69 de referencia.
- **Descartadas:** el titular de prensa no se aleja de la prevalencia del tema migrante; risas + insulto ya lo predice bien el modelo; *hijos de puta* en plural sigue siendo mayoritariamente no-odio; el estereotipo doméstico tiene n = 7.
- **Ojo:** n es pequeño en todas (13-44 tweets) y los patrones se escribieron mirando errores de este mismo train. Las tasas de la tabla están sesgadas a favor de las hipótesis (sección 4).

## 3. Validación cruzada: stacking de 5 modelos (3 semillas)
Con `extra_lex` cambia el bloque de léxico (45 columnas en vez de 38), así que los 5 modelos base se recalculan (caché en `oof/base+errores/`).

In [5]:
dE = tb.Datos(extra_lex=[ev.variables()])
dG = tb.Datos(extra_lex=[ev.variables(generica=True)])
print(dE.tag, dE.L_TR.shape, '|', dG.tag)
def cv5(dd):
    return tb.evaluar(dd, {s: tb.stacking(dd, B5, s, con_test=False)[0] for s in tb.SEEDS})
R = {'E0 · stacking 5 modelos': cv5(d),
     'E10b · + H12-H17': cv5(dE),
     'E10b genérica (sin palabras copiadas de tweets)': cv5(dG)}
for c in ev.COLUMNAS:
    R[f'  sin {c}'] = cv5(tb.Datos(extra_lex=[ev.variables(columnas=[x for x in ev.COLUMNAS if x != c])]))
tb.tabla(R)

base+errores (1731, 45) | base+errores-gen


,F1 HS,F1 TR,F1 AG,F1 medio,±,EMR,F1 medio por semilla
Configuración,,,,,,,
E0 · stacking 5 modelos,0.8038,0.8345,0.7994,0.8126,0.0021,0.7757,0.815 / 0.810 / 0.813
E10b · + H12-H17,0.8183,0.8424,0.8120,0.8242,0.0011,0.7861,0.826 / 0.824 / 0.823
E10b genérica (sin palabras copiadas de tweets),0.8131,0.8391,0.8077,0.8200,0.0035,0.7849,0.822 / 0.823 / 0.815
sin posesivo_fem_sin_eres,0.8135,0.8326,0.8077,0.8179,0.0016,0.7812,0.818 / 0.816 / 0.820
sin negacion_fem,0.8192,0.8427,0.8117,0.8245,0.0018,0.7857,0.827 / 0.824 / 0.823
sin arabe_hostil,0.8205,0.8436,0.8127,0.8256,0.0018,0.7884,0.828 / 0.824 / 0.825
sin hostilidad_migrante,0.8193,0.8433,0.8120,0.8249,0.0021,0.7863,0.828 / 0.824 / 0.823
sin expulsion_migrante,0.8195,0.8428,0.8114,0.8245,0.0010,0.7868,0.825 / 0.823 / 0.825
sin demostrativo_grupo,0.8181,0.8420,0.8094,0.8232,0.0013,0.7845,0.824 / 0.824 / 0.821


**Cada modelo base por separado** (sin stacking):

In [6]:
R1 = {}
for b in B5:
    R1[f'{b}'] = tb.evaluar(d, {s: tb.oof(d, b, s) for s in tb.SEEDS})
    R1[f'{b} + H12-H17'] = tb.evaluar(dE, {s: tb.oof(dE, b, s) for s in tb.SEEDS})
tb.tabla(R1)

,F1 HS,F1 TR,F1 AG,F1 medio,±,EMR,F1 medio por semilla
Configuración,,,,,,,
CNB,0.7509,0.7794,0.7086,0.7463,0.0023,0.7090,0.749 / 0.744 / 0.746
CNB + H12-H17,0.7509,0.7794,0.7086,0.7463,0.0023,0.7090,0.749 / 0.744 / 0.746
LR,0.7924,0.8246,0.7899,0.8023,0.0020,0.7499,0.803 / 0.805 / 0.800
LR + H12-H17,0.8054,0.8303,0.8095,0.8151,0.0018,0.7630,0.816 / 0.813 / 0.817
SVM,0.7890,0.8279,0.7882,0.8017,0.0009,0.7527,0.803 / 0.801 / 0.801
SVM + H12-H17,0.8079,0.8339,0.8096,0.8171,0.0023,0.7695,0.820 / 0.814 / 0.817
NB-LR,0.7916,0.8231,0.7883,0.8010,0.0009,0.7578,0.801 / 0.802 / 0.800
NB-LR + H12-H17,0.8118,0.8310,0.8028,0.8152,0.0027,0.7681,0.815 / 0.812 / 0.819
Powerset,0.7937,0.8284,0.7889,0.8037,0.0021,0.7793,0.806 / 0.804 / 0.801


## 4. ¿Cuánto de la mejora es sesgo de selección?
Las reglas se escribieron leyendo los errores OOF del mismo train con que se mide la CV, así que parte de la mejora puede ser "memorizar" esos tweets. Dos controles:
1. **Versión genérica** (tabla de arriba): sin las palabras tomadas de tweets concretos (*flojo, lapidar, subfusil, tus queridos, mina, morra...*). Si mantiene casi toda la mejora, la señal es general.
2. **Frecuencia en el test**: el test no se usa para ajustar nada, pero se puede contar con qué frecuencia aparece cada patrón. Si un patrón aparece mucho menos en el test que en el train (por tweet), probablemente está ajustado a la redacción concreta de tweets del train.

In [7]:
FT = pd.DataFrame([ev.fila(t) for t in d.TXT_TE])
pd.DataFrame({'train (n)': F.sum(), 'test (n)': FT.sum(),
              'train (% tweets)': 100 * F.mean(), 'test (% tweets)': 100 * FT.mean(),
              'test / train': FT.mean() / F.mean()}).round(2)

,train (n),test (n),train (% tweets),test (% tweets),test / train
posesivo_fem_sin_eres,23,17,1.33,1.47,1.11
negacion_fem,13,3,0.75,0.26,0.35
arabe_hostil,44,15,2.54,1.30,0.51
hostilidad_migrante,29,11,1.68,0.95,0.57
expulsion_migrante,14,4,0.81,0.35,0.43
demostrativo_grupo,28,9,1.62,0.78,0.48
demostrativo_fem,25,21,1.44,1.82,1.26


## 5. Equivalente de E8b (bagging de 5 semillas, 5 modelos + LightGBM)

In [8]:
for s in S5:
    for b in B6:
        tb.oof(dE, b, s); tb.pred_test_folds(dE, b, s)
tb.log('OOF y testfolds de las 5 semillas listos')
R8 = {'E6 · bagging 5 modelos': tb.evaluar(d, {s: tb.stacking_bagged(d, B5, S5, s, con_test=False)[0] for s in tb.SEEDS}),
      'E6 + H12-H17': tb.evaluar(dE, {s: tb.stacking_bagged(dE, B5, S5, s, con_test=False)[0] for s in tb.SEEDS}),
      'E8b · bagging 5 modelos + LGBM': tb.evaluar(d, {s: tb.stacking_bagged(d, B6, S5, s, con_test=False)[0] for s in tb.SEEDS}),
      'E8b + H12-H17': tb.evaluar(dE, {s: tb.stacking_bagged(dE, B6, S5, s, con_test=False)[0] for s in tb.SEEDS})}
tb.tabla(R8)

[    10s] OOF y testfolds de las 5 semillas listos


,F1 HS,F1 TR,F1 AG,F1 medio,±,EMR,F1 medio por semilla
Configuración,,,,,,,
E6 · bagging 5 modelos,0.8092,0.8368,0.7988,0.8150,0.0007,0.7803,0.814 / 0.815 / 0.815
E6 + H12-H17,0.8245,0.8458,0.8144,0.8283,0.0020,0.7891,0.826 / 0.831 / 0.828
E8b · bagging 5 modelos + LGBM,0.8071,0.8405,0.8062,0.8179,0.0012,0.7785,0.819 / 0.818 / 0.816
E8b + H12-H17,0.8211,0.8497,0.8188,0.8299,0.0014,0.7876,0.829 / 0.829 / 0.832


**Cuántas filas del test cambian** frente a E8b (umbral 0,5; no se genera ningún archivo de envío):

In [9]:
Pt8 = np.mean([tb.stacking_bagged(d, B6, S5, s)[1] for s in tb.SEEDS], 0)
PtE = np.mean([tb.stacking_bagged(dE, B6, S5, s)[1] for s in tb.SEEDS], 0)
Yt8, YtE = tb.binarizar(Pt8), tb.binarizar(PtE)
print('Filas del test distintas de E8b:', int((Yt8 != YtE).any(1).sum()),
      '| por etiqueta:', dict(zip(tb.LABELS, (Yt8 != YtE).sum(0))))
print('Prevalencia train              :', dict(zip(tb.LABELS, d.Y.mean(0).round(3))))
print('Prevalencia predicha E8b       :', dict(zip(tb.LABELS, Yt8.mean(0).round(3))))
print('Prevalencia predicha E8b+H12-17:', dict(zip(tb.LABELS, YtE.mean(0).round(3))))

Filas del test distintas de E8b: 52 | por etiqueta: {'HS': np.int64(34), 'TR': np.int64(17), 'AG': np.int64(35)}
Prevalencia train              : {'HS': np.float64(0.437), 'TR': np.float64(0.3), 'AG': np.float64(0.382)}
Prevalencia predicha E8b       : {'HS': np.float64(0.403), 'TR': np.float64(0.299), 'AG': np.float64(0.369)}
Prevalencia predicha E8b+H12-17: {'HS': np.float64(0.399), 'TR': np.float64(0.298), 'AG': np.float64(0.37)}


## 6. Opcional: normalización que repara la vocal acentuada borrada
El corpus perdió los caracteres no ASCII (*cállate* → `cllate`, *imbécil* → `imbcil`). `ev.normalizar_tildes` aprende **solo del train** un mapeo de las palabras imposibles en español (grupos de consonantes que no existen) a la forma con una vocal insertada más frecuente (`cllate` → `callate`, `vdeo` → `video`, `cabrn` → `cabron`).

In [10]:
dN = tb.Datos(normalizador=ev.normalizar_tildes)
dNE = tb.Datos(normalizador=ev.normalizar_tildes, extra_lex=[ev.variables()])
print('Palabras en el mapeo:', len(ev._MAPEO), '| ejemplos:', list(ev._MAPEO.items())[:12])
tb.tabla({'E0 · stacking 5 modelos': R['E0 · stacking 5 modelos'], 'E0 + normalizar_tildes': cv5(dN),
          'E10b · + H12-H17': R['E10b · + H12-H17'], 'E10b + normalizar_tildes': cv5(dNE)})

Palabras en el mapeo: 38 | ejemplos: [('cllate', 'callate'), ('estmos', 'estamos'), ('slo', 'solo'), ('vdeo', 'video'), ('ests', 'estos'), ('cmo', 'como'), ('ejrcito', 'ejercito'), ('cnico', 'cinico'), ('cagn', 'cagon'), ('cmplice', 'complice'), ('estpida', 'estupida'), ('estn', 'estan')]


,F1 HS,F1 TR,F1 AG,F1 medio,±,EMR,F1 medio por semilla
Configuración,,,,,,,
E0 · stacking 5 modelos,0.8038,0.8345,0.7994,0.8126,0.0021,0.7757,0.815 / 0.810 / 0.813
E0 + normalizar_tildes,0.8061,0.8352,0.8001,0.8138,0.0021,0.7778,0.816 / 0.811 / 0.815
E10b · + H12-H17,0.8183,0.8424,0.8120,0.8242,0.0011,0.7861,0.826 / 0.824 / 0.823
E10b + normalizar_tildes,0.8186,0.8417,0.8109,0.8237,0.0032,0.7859,0.828 / 0.820 / 0.823


## 7. Conclusión: mejora clara en la CV, pero **inflada por sesgo de selección**; candidato con cautela
- **Stacking de 5 modelos (3 semillas):** 0,8126 → **0,8242** (+0,0116), mejor en las 3 semillas (0,826 / 0,824 / 0,823 frente a 0,815 / 0,810 / 0,813). Supera con holgura el criterio de adopción (+0,003). Suben las 3 etiquetas, sobre todo HS y AG. Cada modelo lineal por separado gana +1,1 a +1,5 puntos (LR: 0,8023 → 0,8151); CNB no cambia porque no usa el bloque de léxico.
- **Equivalente de E8b** (bagging de 5 semillas, 5 modelos + LightGBM): 0,8179 → **0,8299** (+0,012), mejor en las 3 semillas (0,829 / 0,829 / 0,832 frente a 0,819 / 0,818 / 0,816). E6 + H12-H17 da 0,8283. Cambian 52 filas del test frente a E8b, con prevalencias predichas casi iguales.
- **Ablación:** la mayor parte de la mejora la aportan **H12 (posesivo; quitarla cuesta −0,006)**, **H17 (demostrativo femenino, −0,0016)** y **H16 (demostrativo + grupo, −0,001)**. H13 y H15 no aportan más que el ruido entre semillas. Quitar `arabe_hostil` (H14) incluso sube un poco (0,8256). La hipótesis que parecía más fuerte en la tabla de verificación es la que menos generaliza dentro de la CV.
- **Sesgo de selección (lo más importante para el informe):** las reglas se escribieron leyendo los errores OOF de este mismo train, así que la CV sobreestima la mejora. Hay dos señales:
  1. La versión **genérica** (sin palabras copiadas de tweets concretos) conserva solo **+0,0074** (0,8200), unas dos terceras partes de la mejora, y varía más entre semillas.
  2. En el **test** (solo conteos, sin etiquetas), los patrones más hechos a medida aparecen, por tweet, entre 0,35 y 0,57 veces su frecuencia en el train (`negacion_fem`, `expulsion_migrante`, `demostrativo_grupo`, `arabe_hostil`). En cambio, `posesivo_fem` y `demostrativo_fem`, las que más aportan, aparecen con una frecuencia igual o mayor. En el leaderboard cabe esperar una mejora menor que en la CV, concentrada en H12 y H17.
- **Normalización de tildes:** apenas +0,001 sobre E0 y nada sobre E10b, porque los n-gramas de caracteres y los patrones `c(a)?llate` ya cubren esas grafías. No se adopta.
- **Lectura general:** se confirma la conclusión de E5. Lo que mueve el F1 en este corpus son patrones de **contexto** del español (posesivos, demostrativos, concordancia), no listas de palabras. Si se envía, conviene usar la versión completa (la que se diseñó primero) y no la mejor de la ablación, para no sumar más selección sobre la CV.
- **Uso:** `import ext_variables as ev; d = tb.Datos(extra_lex=[ev.variables()])`. Las OOF y los `testfolds` de las 6 bases × 5 semillas quedan en caché en `oof/base+errores/`.